# 다이캐스팅 공정 데이터 기반 품질 예측 프로젝트

- 다이캐스팅은 용융 금속을 금형에 고압으로 주입하여 정밀한 제품을 생산하는 공정이다.
- 생산 효율을 높이고 불량품을 낮추기 위해 공정 변수 및 센서 데이터를 분석해서 불량 유형을 자동 예측하는 머신러닝 모델을 개발해야함.
- 실시간 공정 데이터와 품질 검사를 연계하여 품질 예측 시스템을 구축하고자 함.

# 비즈니스 문제 정의

### 현재 상황

- 불량품이 발생해도 육안 검사에 의존하여 판정 기준의 주관성과 검사 속도의 한계로 생산성이 저하됨.
- 불량 발생 원인을 추적하기 힘들어 공정 개선 및 문제 해결이 어려움.
- 공정 데이터와 품질 검사를 효과적으로 매핑하지 못하여 실시간 품질 관리 및 재발 방지 대책이 부족한 상황.

### 분석 목표

- 다이캐스팅 공정 과정에서 발생하는 다양한 불량 유형을 자동으로 예측해주는 머신 러닝 모델 개발
- 공정 데이터와 센서 데이터를 면밀히 분석해 불량 여부와의 관계성 파악
- 불량 발생의 주요 원인을 분석해서 공정의 최적화를 지원
- 실시간 품질 예측 체계를 구축하여 조기 경보 시스템 도입 및 불량률 감소

### 비즈니스 산출물

1) 불량 발생의 주요 원인을 분석하고 이를 시각화하여 제시
2) 불량 유형을 자동으로 예측해주는 머신러닝 모델 제시

# 데이터셋 구성

- **공정(Process)데이터**
1) Shot ID : 주조 샷 고유 식별자
2) Injection Speed : 용탕 주입 속도 (m/s)
3) Die Temperature : 금형 온도
4) Casting Pressure : 주조 압력 (bar)
5) Cooling Time: 냉각 시간 (s)

- **센서(Sensor) 데이터**
1) Mold Temp Sensor : 금형 내 센서 온도 (°C)
2) Hydraulic Pressure : 유압 압력 (bar)
3) Vibration Sensor : 진동값 (Hz)
4) Flow Rate Sensor : 유량 (L/min)

- **불량(Defects) 데이터**
1) Defect Type : 발생한 불량 유형 (미성형, 박리, 기공, 평탄, 개재물 등)
2) Defect Status : 양품(0) / 불량(1) 여부

In [1]:
# ============================================================
# 공정(Process) 주요 컬럼 설명
# ============================================================
# • Shot ID                       : 주조 번호 (붕어빵 틀에 반죽을 한 번 붓고 뚜껑을 닫는 행위에 관한 일련번호)
# • Injection Speed               : 주입 속도 (쇳물을 금형에 얼마나 빨리 밀어 넣는가)
# • Die Temperature               : 금형 온도 (금형 틀이 어느정도의 온도인가)
# • Casting Pressure              : 주조 압력 (쇳물을 다 채운 뒤 강하게 누르는 힘)
# • Cooling Time                  : 냉각 시간 (액체 상태인 금속이 고체가 될 떄까지 기다리는 힘)


# ============================================================
# 공정(Process) 컬럼별 분석 관점
# ============================================================
# • Injection Speed               : 주입 속도가 너무 느리면 쇳물이 금형으로 가다가 굳어버리고, 너무 빠르면 사방으로 튀어서 속에 공기 방울(기공)이 생길 수 있다.
# • Die Temperature               : 금형 온도가 너무 낮으면 쇳물이 일찍 굳고 너무 높으면 금형 수명이 줄거나 제품이 달라붙는 현상이 생긴다.
# • Casting Pressure              : 압력이 너무 낮으면 기공이 발생할 수 있고, 너무 높으면 금형 손상 및 제품 변형이 일어날 수 있다.
# • Cooling Time                  : 냉각 시간이 너무 짧으면 탈형 시 변형이 오고, 너무 느리면 사이클 타임이 늘어나 생산성이 떨어진다.


# ============================================================
# 센서(Sensor) 주요 컬럼 설명
# ============================================================
# • Mold Temp Sensor               : 금형 내부 온도 (Die Temperature가 전체의 겉 온도라면, 이거는 틀 안쪽 깊숙한 곳의 실시간 온도이다)
# • Hydraulic Pressure             : 유압 압력 (bar) (쇳물을 밀어내기 위해 기계 팔(실린더)이 쓰는 기름의 힘 혈압과 비슷하다고 볼 수 있음)
# • Vibration Sensor               : 진동값 (Hz) (기계가 작동할 떄의 떨림)
# • Flow Rate Sensor               : 유량 (L/min) (틀을 식히기 위한 냉각수가 흐르는 속도를 나타냄)


# ============================================================
# 센서(Sensor) 컬럼별 분석 관점
# ============================================================
# • Mold Temp Sensor               : 쇳물이 들어올 때 순간 확 올랐다가 식으면서 툭 떨어지는 경향을 보인다
# • Hydraulic Pressure             : 유압 압력은 혈압과 비슷하다고 생각할 수 있다. 유압 압력이 일정해야 쇳물을 쏘아 주는 힘이 균일해진다.
# • Vibration Sensor               : 평소보다 많이 흔들린다면 나사가 풀렸거나, 부품이 마모됐거나, 쇳물이 튀어서 어딘가 걸렸다는 신호일 수 있다.
# • Flow Rate Sensor               : 유량이 적으면 틀이 식지 않아서 제품이 녹아내릴 수 있다.


# ============================================================
# 불량(Defects) 주요 컬럼 설명
# ============================================================
# • Defect Type                   : 발생한 불량 유형
# • Defect Status                 : 양품(0) / 불량(1) 여부


# ============================================================
# 불량(Defects) 유형 설명
# ============================================================
# • 미성형 (Underfill/Short Shot)  : 반죽이 모자라 다 완성되지 못한 붕어빵 (주입 속도가 느리거나, 금형 온도가 낮을 때 주로 발생) -> Injection Speed, Mold Temp Sensor
# • 박리 (Lamination/Peeling)      : 금속 층이 서로 제대로 붙지 않고 겹쳐진 상태 (쇳물이 들어갈 때 너무 차갑거나, 불순물이 섞여 층이 생기면 발생) -> Mold Temp Sensor, Flow Rate Sensor
# • 기공 (Porosity/Gas Hole)       : 공기 방울이 송송 뚫린 상태(주조 압력이 너무 낮거나, 주입 속도가 너무 빨라서 공기가 들어갔을 때 생김) -> Casting Pressure, Injection Speed
# • 평탄 (Flatness Error)          : 바닥이 평평하지 않고 휘어진 붕어빵 (냉각 시간이 너무 짧거나, 냉각수가 골고루 잘 흐르지 못했을 때 발생) -> Cooling Time, Flow Rate Sensor
# • 개재물 (Inclusion)             : 반죽에 검은 가루나 돌멩이가 들어간 것 (재료 관리가 안되었거나, 필터에 문제가 있을 때 발생) -> Flow Rate Sensor

---
# 1.1 필요 라이브러리 및 폰트 로드

In [2]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform

# 시각화
import matplotlib.pyplot as plt

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)


라이브러리 로드 완료!
한글 폰트 설정 완료!


# 1.2 데이터 로드

In [3]:
# 1. 원본 데이터 로드
df_original = pd.read_csv("../data/DieCasting_Quality_Raw_Data.csv", header=[0,1])

# 2. Product-2 유형의 데이터 추출
df_original_product_2 = df_original[df_original[('Process','Product_Type')] == 2].reset_index(drop=True)

# 2-1. 작업 확인용
print(f"product_type 2의 데이터 개수: {len(df_original_product_2)}개")
print('='*30)

# 2-2. csv 파일로 저장
# 인덱스는 새로운 컬럼으로 추가하지 않음
df_original_product_2.to_csv("../common-file/Data_of_product-2.csv", index=False)


# 3. 컬럼명 중 첫번째 행을 기준으로 컬럼 분리
process_cols = [col for col in df_original_product_2.columns if col[0] == 'Process']
sensor_cols = [col for col in df_original_product_2.columns if col[0] == 'Sensor']
defects_cols = [col for col in df_original_product_2.columns if col[0] == 'Defects']

# 4. 첫 번째 컬럼을 기준으로 분리된 데이터 프레임 생성
df_process_2 = df_original_product_2[process_cols].copy()
df_sensor_2 = df_original_product_2[sensor_cols].copy()
df_defects_2 = df_original_product_2[defects_cols].copy()

# 4-1. 작업 확인용
for name, d in [('Process', df_process_2), ('Sensor', df_sensor_2), ('Defects', df_defects_2)]:
    print(f"{name}: {d.shape[1]}개 컬럼, {d.shape[0]}개")

# 5. 두 번째 행에 있는 컬럼명만 사용하도록 변경
# get_level_values(가져오고 싶은 레벨 인덱스) - level 0: 첫번째 행, level 1: 두번째 행
# 왼쪽 .columns: 컬럼명을 바꾸기 위한 설정
# 오른쪽 .columns: 기존 다중레벨 컬럼 가져오기 → get_level_values(1)로 Level 1만 추출
df_process_2.columns = df_process_2.columns.get_level_values(1)
df_sensor_2.columns = df_sensor_2.columns.get_level_values(1)
df_defects_2.columns = df_defects_2.columns.get_level_values(1)

product_type 2의 데이터 개수: 3328개
Process: 17개 컬럼, 3328개
Sensor: 14개 컬럼, 3328개
Defects: 26개 컬럼, 3328개


---
---
# 2 데이터 전처리 (Data Preprocessing)

### 2.1 중복 데이터 확인

In [4]:
# 1. 전체 행에 대한 중복값 확인 -> 완전히 동일한 행이 있는지 확인하는 용도 -> 없음
print(f"전체 행에 대한 데이터 중복값의 개수 : {df_original_product_2.duplicated().sum()}건")

# 2. ID 기준으로 중복값 확인 -> 설비 ID 중복 확인용 -> 없음
print(f"(ID 기준) 중복값의 개수 : {df_process_2.duplicated(subset=["id"]).sum()}건")


전체 행에 대한 데이터 중복값의 개수 : 0건
(ID 기준) 중복값의 개수 : 0건


---
### 2.2 데이터 타입 확인

In [5]:
# 1. 각 데이터 프레임별 컬럼의 데이터 형식 확인
print('='*30)
print("<Process 관련 데이터프레임 정보>")
print('='*30)
display(df_process_2.dtypes)

print('='*30)
print("<Sensor 관련 데이터프레임 정보>")
print('='*30)
display(df_sensor_2.dtypes)

print('='*30)
print("<Defects 관련 데이터프레임 정보>")
print('='*30)
display(df_defects_2.dtypes)

<Process 관련 데이터프레임 정보>


id                       int64
Product_Type             int64
Shot                     int64
Velocity_1             float64
Velocity_2             float64
Velocity_3             float64
High_Velocity          float64
Cylinder_Pressure        int64
Rapid_Rise_Time        float64
Biscuit_Thickness        int64
Clamping_Force           int64
Cycle_Time             float64
 Pressure_Rise_Time    float64
Casting_Pressure         int64
Spray_Time             float64
Spray_1_Time           float64
Spray_2_Time           float64
dtype: object

<Sensor 관련 데이터프레임 정보>


Melting_Furnace_Temp    float64
Air_Pressure            float64
Air_Pressure_Min          int64
Air_Pressure_Max          int64
Coolant_Temp            float64
Coolant_Temp_Min          int64
Coolant_Temp_Max          int64
Coolant_Pressure        float64
Factory_Temp            float64
Factory_Temp_Min        float64
Factory_Temp_Max        float64
Factory_Humidity        float64
Factory_Humidity_Min    float64
Factory_Humidity_Max    float64
dtype: object

<Defects 관련 데이터프레임 정보>


Short_Shot_1       int64
Bubble_1           int64
Exfoliation_1      int64
Blow_Hole_1        int64
Stain_1            int64
Dent_1             int64
Deformation_1      int64
Contamination_1    int64
Impurity_1         int64
Crack_1            int64
Scratch_1          int64
Buring_Mark_1      int64
Inclusions_1       int64
Short_Shot_2       int64
Bubble_2           int64
Exfoliation_2      int64
Blow_Hole_2        int64
Stain_2            int64
Dent_2             int64
Deformation_2      int64
Contamination_2    int64
Impurity_2         int64
Crack_2            int64
Scratch_2          int64
Buring_Mark_2      int64
Inclusions_2       int64
dtype: object

---
### 2.3 컬럼 정제

#### 2.3.1. 공백 제거 및 컬럼명 소문자 통일

In [6]:
# 1. 공백 제거 및 컬럼명 소문자 통일
df_process_2.columns = df_process_2.columns.astype("str").str.strip().str.lower()
df_sensor_2.columns = df_sensor_2.columns.astype("str").str.strip().str.lower()
df_defects_2.columns = df_defects_2.columns.astype("str").str.strip().str.lower()

In [7]:
# 2. Sensor 관련 데이터프레임에서 _min/_max로 끝나는 컬럼 찾아서 삭제
minmax_cols = [c for c in df_sensor_2.columns if c.endswith("_min") or c.endswith("_max")]
df_cleaned_sensor_2 = df_sensor_2.drop(columns=minmax_cols)


print(f"삭제한 '_min/_max'관련 컬럼 수: {len(minmax_cols)}개")
print(f"삭제 전 df_sensor_2 shape: {df_sensor_2.shape}")
print(f"삭제 후 df_sensor_2 shape: {df_cleaned_sensor_2.shape}")

삭제한 '_min/_max'관련 컬럼 수: 8개
삭제 전 df_sensor_2 shape: (3328, 14)
삭제 후 df_sensor_2 shape: (3328, 6)


#### 2.3.2 불량 유형 고유값 통일 + cavity 1,2 불량 유형 통합

In [8]:
# 1. 1(고장)이 아닌 유형의 값이 존재하는 컬럼
target_columns = [
    "short_shot_1", "exfoliation_1", "blow_hole_1", "stain_1", "deformation_1",
    "short_shot_2", "bubble_2", "exfoliation_2", "blow_hole_2", "deformation_2"
]
# 2. 1보다 큰 값을 모두 1로 변환
df_defects_2[target_columns] = df_defects_2[target_columns].clip(upper=1)

# 3. 결과 확인 <- 모든 컬럼의 고유값이 [0, 1]로 정리되었는지 확인하기 위함 
print("<1보다 큰 값은 1로 변환한 후, 고유값 확인>")
for col in df_defects_2:
    print(f"{col}: {df_defects_2[col].unique()}")

<1보다 큰 값은 1로 변환한 후, 고유값 확인>
short_shot_1: [0 1]
bubble_1: [0 1]
exfoliation_1: [0]
blow_hole_1: [0 1]
stain_1: [0 1]
dent_1: [0 1]
deformation_1: [0]
contamination_1: [0 1]
impurity_1: [0 1]
crack_1: [0 1]
scratch_1: [0 1]
buring_mark_1: [0 1]
inclusions_1: [0]
short_shot_2: [0 1]
bubble_2: [0]
exfoliation_2: [0]
blow_hole_2: [0 1]
stain_2: [0]
dent_2: [0 1]
deformation_2: [0]
contamination_2: [0 1]
impurity_2: [0 1]
crack_2: [0 1]
scratch_2: [0]
buring_mark_2: [0]
inclusions_2: [0 1]


In [9]:
# 4. Cavity 1,2 불량 유형 통합
df_defects_clean_step1 =df_defects_2.copy()

# 4-1. cavity 1/2 컬럼 분리
cavity_1 =df_defects_clean_step1[[c for c in df_defects_clean_step1.columns if c.endswith("_1")]].copy()
cavity_2 =df_defects_clean_step1[[c for c in df_defects_clean_step1.columns if c.endswith("_2")]].copy()

# 4-2. 컬럼명 통일: short_shot_1 -> short_shot
cavity_1.columns = [c.replace("_1", "") for c in cavity_1.columns]
cavity_2.columns = [c.replace("_2", "") for c in cavity_2.columns]

# 4-3. 제대로 분리되었는지 확인
print("cavity_1 shape:", cavity_1.shape)
print("cavity_2 shape:", cavity_2.shape)
print("cavity_1, cavity_2 컬럼이 동일한가?", set(cavity_1.columns) == set(cavity_2.columns))

cavity_1 shape: (3328, 13)
cavity_2 shape: (3328, 13)
cavity_1, cavity_2 컬럼이 동일한가? True


In [10]:
# 4-4. OR 방식으로 통합: 둘 중 하나라도 1이 있다면 1이 입력됨
defects_merged = ((cavity_1 + cavity_2) > 0).astype(int)   # (cavity_1 | cavity_2) 도 가능
df_cleaned_defects_2_step2 = defects_merged

print("통합 전 df_defects shape:",df_defects_2.shape)
print("통합 후 df_cleaned_defects_2 shape:",df_cleaned_defects_2_step2.shape)
df_cleaned_defects_2_step2.head(10)

통합 전 df_defects shape: (3328, 26)
통합 후 df_cleaned_defects_2 shape: (3328, 13)


,short_shot,bubble,exfoliation,blow_hole,stain,dent,deformation,contamination,impurity,crack,scratch,buring_mark,inclusions
0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0,0


In [11]:
# 4-5. 전체 셀 기준(모든 defect 칸을 다 펼쳐서 0/1/2 비율)
flat = df_cleaned_defects_2_step2.to_numpy().ravel()
dist_all = pd.Series(flat).value_counts(normalize=True).reindex([0,1,2], fill_value=0)
print("<전체 셀 기준 0/1/2 비율>")
print((dist_all * 100).round(2).astype(str) + "%")

<전체 셀 기준 0/1/2 비율>
0    97.62%
1     2.38%
2      0.0%
Name: proportion, dtype: object


#### 2.3.3 불량 유형 분리
- 표면 불량(surface_defect) : 육안으로 확인 가능하지만, 금속의 분리나 갈라짐은 없는 불량        
    - stain(얼룩), dent(찍힘), scratch, burning_mark(소착)

- 구조 불량(structural_defect): 육안으로 금속의 분리·갈라짐이 보이거나, 제품의 강도·기능에 직접 영향을 줄 수 있는 불량 
    - short_shot(미성형), bubble(기포), blow_hole(기공), deformation(변형), crack(균열), exfoliation(박리)

- 이물질 포함 불량(contamination_defect): 원래 포함되면 안 되는 외부 물질이 들어간 불량
    - contamination(오염), impurity(이물), inclusions(개재물)

In [12]:
# 불량 유형 범주화
# 같은 유형의 불량으로 구분된 불량유형이 하나의 shot에 동시에 존재하더라도 1로 처리됨
df_defects_groups_2 = pd.DataFrame(index=df_cleaned_defects_2_step2.index)

df_defects_groups_2["surface_defect"] = ( # 표면과 관련된 불량
    df_cleaned_defects_2_step2.reindex(columns=["stain", "dent", "scratch", "burning_mark"], fill_value=0).max(axis=1)
)

df_defects_groups_2["structural_defect"] = ( # 구조와 관련된 불량
    df_cleaned_defects_2_step2.reindex(columns=["short_shot", "bubble", "blow_hole", "deformation", "crack", "exfoliation"], fill_value=0).max(axis=1)
)

df_defects_groups_2["contamination_defect"] = ( # 이물질이 포함된 불량
    df_cleaned_defects_2_step2
    .reindex(columns=["contamination", "impurity", "inclusions"], fill_value=0)
    .max(axis=1)
)


print("df_defects_groups_2 shape:", df_defects_groups_2.shape)
print('='*30)
print("<각 유형별 불량 개수>")
display(df_defects_groups_2.sum().sort_values(ascending=False)) # 각 유형별 불량 개수
print('='*30)
df_defects_groups_2.info() # Non-Null인 행의 개수 확인

df_defects_groups_2 shape: (3328, 3)
<각 유형별 불량 개수>


structural_defect       778
surface_defect          196
contamination_defect     19
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3328 entries, 0 to 3327
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   surface_defect        3328 non-null   int64
 1   structural_defect     3328 non-null   int32
 2   contamination_defect  3328 non-null   int32
dtypes: int32(2), int64(1)
memory usage: 52.1 KB


#### 2.3.4 Process 관련 데이터의 컬럼 정리
- id, shot: 새로운 식별자("shot_key")에 통합된 정보이므로 삭제
- product_type: 제품유형 1에 대한 데이터만 처리 중이므로 삭제

In [13]:
# 1. "shot_key" 컬럼 생성
df_cleaned_process_2 =df_process_2.copy()
df_cleaned_process_2["shot_key"] =df_cleaned_process_2["id"].astype(str) + "_" + df_cleaned_process_2["shot"].astype(str)

# 2. id, Shot, product_type 컬럼 삭제
df_cleaned_process_2 =df_cleaned_process_2.drop(columns=["id", "shot", "product_type"])

# 3. "shot_key" 컬럼을 맨 앞으로 이동
cols = ["shot_key"] + [c for c in df_cleaned_process_2.columns if c != "shot_key"]
df_cleaned_process_2 =df_cleaned_process_2[cols]

# 4. "shot_key"가 중복인 값이 있는지 확인
dup =df_cleaned_process_2["shot_key"][df_cleaned_process_2["shot_key"].duplicated(keep=False)]
print("중복인 값의 개수:", dup.shape[0])

중복인 값의 개수: 0


In [14]:
# 5. shot_key 컬럼이 추가된 전체 데이터프레임 제작
df_clean_product_2 = pd.concat([df_cleaned_process_2, df_cleaned_sensor_2, df_defects_groups_2], axis=1)

print("df_clean_product_2 shape:", df_clean_product_2.shape)
df_clean_product_2.head()

df_clean_product_2 shape: (3328, 24)


,shot_key,velocity_1,velocity_2,velocity_3,high_velocity,cylinder_pressure,rapid_rise_time,biscuit_thickness,clamping_force,cycle_time,pressure_rise_time,casting_pressure,spray_time,spray_1_time,spray_2_time,melting_furnace_temp,air_pressure,coolant_temp,coolant_pressure,factory_temp,factory_humidity,surface_defect,structural_defect,contamination_defect
0,4207011_11,0.156,0.166,0.192,2.723,265,0.012,20,357,36.6,0.041,595,12.5,2.0,2.2,671.6,6.5,26.1,2.70,33.2,57.4,0,0,0
1,4208012_12,0.157,0.166,0.204,2.730,264,0.014,19,359,36.5,0.040,594,12.5,2.0,2.2,672.1,6.4,26.2,2.71,33.3,57.0,0,0,0
2,4209013_13,0.156,0.170,0.204,2.715,265,0.012,18,361,36.5,0.041,595,12.5,2.0,2.2,672.4,6.4,26.2,2.70,33.5,56.7,0,0,0
3,4210014_14,0.154,0.170,0.202,2.717,264,0.011,20,364,36.5,0.042,595,12.5,2.0,2.2,672.4,6.4,26.2,2.70,33.5,56.7,0,0,0
4,4211015_15,0.146,0.160,0.198,2.684,264,0.012,20,357,36.5,0.042,595,12.5,2.0,2.2,672.4,6.3,26.2,2.71,33.6,56.4,0,0,0


---
### 2.4 결측값 확인

In [15]:
# 1. 데이터프레임 내 결측값 확인 -> 없음
na_count = df_clean_product_2.isna().sum().sort_values(ascending=False)
na_cols = na_count[na_count > 0].sort_values(ascending=False)

print("<그룹별 결측치(전체 결측값의 개수)>")
print(pd.Series({
    "Process": df_cleaned_process_2.isna().sum().sum(),
    "Sensor":  df_cleaned_sensor_2.isna().sum().sum(),
    "Defects": df_defects_groups_2.isna().sum().sum()
}))
print()

print('='*30)
print("<결측치가 있는 컬럼 및 결측값의 수>")
print(na_cols.sort_values(ascending=False))
print()

<그룹별 결측치(전체 결측값의 개수)>
Process      0
Sensor     180
Defects      0
dtype: int64

<결측치가 있는 컬럼 및 결측값의 수>
factory_humidity    90
factory_temp        90
dtype: int64



#### 2.4.1 머신러닝용 데이터셋 저장 (.csv)

In [16]:
df_clean_product_2.to_csv("../common-file/for_ML_overall_product-2.csv")
df_cleaned_process_2.to_csv("../common-file/for_ML_process_data_product-2.csv")
df_cleaned_sensor_2.to_csv("../common-file/for_ML_sensor_data_product-2.csv")
df_defects_groups_2.to_csv("../common-file/for_ML_defects_data_product-2.csv")

#### 2.4.2 결측치 처리
- 공장 온도/습도 관련 데이터에 결측치가 존재하여 중앙값으로 채움

In [17]:
# 중앙값으로 채움
fill_cols = ["factory_temp", "factory_humidity"]
df_cleaned_sensor_2[fill_cols] = df_cleaned_sensor_2[fill_cols].fillna(df_cleaned_sensor_2[fill_cols].median())

print(df_cleaned_sensor_2[fill_cols].isna().sum())
print("남아있는 결측치의 개수:", df_cleaned_sensor_2[fill_cols].isna().sum().sum())

factory_temp        0
factory_humidity    0
dtype: int64
남아있는 결측치의 개수: 0


---
### 2.5 이상치 처리
- Sensor 관련 데이터 중 공장의 온도/습도 데이터는 이상치를 정의하지 않음
- 온도/습도를 제외한 Sensor/Process 관련 데이터는 아래와 같은 기준으로 이상치 정의
    - "왜도 < |2| 또는 첨도 < |8|" 이라는 조건을 둘 다 만족하는 컬럼은 z-score를 이용해 3σ 방식으로 이상치 정의
    - 두 조건 중 하나라도 만족하지 못한다면 IQR 방식으로 이상치 정의
- 플래깅, 클래핑 기법을 사용해 이상치 처리

#### 2.5.1 Process, Sensor 관련 데이터의 기술통계량 확인

In [18]:
# 1. Process, Sensor 관련 데이터의 기초통계량 확인
def create_statistics_summary(df, df_name, exclude_cols=None):
    
    print(f"\n{'='*80}")
    print(f"⬇️{df_name} 관련 데이터의 기초통계량⬇️")
    print(f"{'='*80}\n")
    
    # 1-1. exclude_cols에 속한 컬럼 제외한 데이터프레임 생성
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')
    
    # 1-2. 기초 통계량
    df_stats = df_copied.describe().T
    
    # 1-3. 고유값의 개수, 왜도, 첨도 추가
    df_stats['Unique'] = df_copied.nunique()
    df_stats['Skewness'] = df_copied.skew()
    df_stats['Kurtosis'] = df_copied.kurtosis()
    
    # 1-4. 컬럼명 한글로 변경
    df_stats.rename(columns={
        'count': '전체 데이터 수', # 결측치가 아닌 값의 개수
        'mean': '평균',
        'std': '표준편차',
        'min': '최솟값',
        '25%': 'Q1의 경계값',
        '50%': '중앙값',
        '75%': 'Q3의 경계값',
        'max': '최댓값',
        'Unique': '고유값의 개수',
        'Skewness': '왜도',
        'Kurtosis': '첨도'
    }, inplace=True)
    
    display(df_stats)
    
    return df_stats

In [19]:
# 2. 함수 실행
stats_df_process_2 = create_statistics_summary(df_cleaned_process_2, '제품 2의 Process', exclude_cols=['shot_key'])
stats_df_sensor_2 = create_statistics_summary(df_cleaned_sensor_2, '제품 2의 Sensor')


⬇️제품 2의 Process 관련 데이터의 기초통계량⬇️



,전체 데이터 수,평균,표준편차,최솟값,Q1의 경계값,중앙값,Q3의 경계값,최댓값,고유값의 개수,왜도,첨도
velocity_1,3328.0,0.154473,0.004838,0.139,0.1515,0.156,0.158,0.162,20,-0.723734,-0.280219
velocity_2,3328.0,0.168620,0.004023,0.158,0.1660,0.168,0.172,0.178,16,0.066497,-0.811232
velocity_3,3328.0,0.202247,0.004953,0.184,0.2000,0.202,0.206,0.216,26,-0.215835,0.367787
high_velocity,3328.0,2.553245,0.071882,2.470,2.5140,2.524,2.538,2.744,155,1.579370,0.745153
cylinder_pressure,3328.0,264.764123,0.756067,247.000,265.0000,265.000,265.000,266.000,6,-15.937863,364.730170
rapid_rise_time,3328.0,0.011660,0.000887,0.009,0.0110,0.012,0.012,0.014,6,-0.645471,2.402679
biscuit_thickness,3328.0,17.589243,1.492139,2.000,17.0000,18.000,19.000,24.000,14,-1.538814,14.104029
clamping_force,3328.0,370.342548,10.160827,346.000,361.0000,375.000,379.000,388.000,24,-0.470378,-1.228546
cycle_time,3328.0,35.704838,2.509634,33.600,35.8000,36.000,36.100,125.900,31,28.736696,1008.900533
pressure_rise_time,3328.0,0.036638,0.002946,0.031,0.0340,0.036,0.040,0.045,14,0.692732,-0.834263



⬇️제품 2의 Sensor 관련 데이터의 기초통계량⬇️



,전체 데이터 수,평균,표준편차,최솟값,Q1의 경계값,중앙값,Q3의 경계값,최댓값,고유값의 개수,왜도,첨도
melting_furnace_temp,3328.0,655.703996,8.494439,635.30,648.70,655.4,662.50,678.10,323,0.273951,-0.647366
air_pressure,3328.0,6.120583,0.677288,4.60,5.60,6.2,6.80,7.10,26,-0.378966,-0.930384
coolant_temp,3328.0,26.923347,0.551408,25.90,26.50,26.8,27.30,28.10,23,0.282971,-0.738899
coolant_pressure,3328.0,2.689742,0.056365,2.58,2.63,2.7,2.74,2.79,22,-0.210328,-1.543214
factory_temp,3328.0,32.571725,1.522960,27.40,31.60,32.0,32.50,37.00,67,1.401438,1.016791
factory_humidity,3328.0,63.190775,6.631947,45.50,61.80,64.3,69.10,72.30,210,-1.083763,0.270639


#### 2.5.2 이상치 정의 및 Flagging 컬럼 추가
- 이상치를 찾은 뒤, (컬럼명)_outlier__flag라 하는 새로운 컬럼을 추가해 이상치 여부 표시
    - 이상치라면 1, 그렇지 않다면 0으로 채워짐

In [20]:
# 1. 함수 정의
def flag_outliers(df, skew_holder = 2, kurt_holder = 8):
    """
    왜도 < |2| AND 첨도 < |8| → Z-score 3σ 방식
    둘 중 하나라도 미충족 → IQR 방식
    """
    
    df_flagged = df.copy()
    numeric_cols = df.select_dtypes(include=["number"]).columns # 숫자형 컬럼만 선택

    
    for column in numeric_cols:
        skew_value = df[column].skew()
        kurt_value = df[column].kurt()

        if abs(skew_value) < skew_holder and abs(kurt_value) < kurt_holder:
            # Z-score; 3σ 방식
            mean = df[column].mean()
            std = df[column].std(ddof=0)
            df_flagged[f'{column}_outlier_flag'] = (
                # "df[column] - mean"(=편차)의 결과에 절댓값을 씌우고, 표준편차의 3배 이상이면 이상치로 정의
                (df[column] - mean).abs() > 3 * std  
            ).astype(int)

        else:
            # IQR 방식
            Q1 = df[column].quantile(0.25)
            Q3 = df[column].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            df_flagged[f'{column}_outlier_flag'] = (
                (df[column] < lower_bound) | (df[column] > upper_bound)
            ).astype(int)

    return df_flagged

In [21]:
# 2. 함수 실행
df_flagged_process_2 = flag_outliers(df_cleaned_process_2)
df_flagged_sensor_2 = flag_outliers(df_cleaned_sensor_2)

print('='*30)
print("<제품 2의 Process 관련 변수의 데이터프레임(컬럼추가)>")
print('='*30)
display(df_flagged_process_2.head(10))

print('='*30)
print("<제품 2의 Sensor 관련 변수의 데이터프레임(컬럼추가)>")
print('='*30)
display(df_flagged_sensor_2.head(10))

<제품 2의 Process 관련 변수의 데이터프레임(컬럼추가)>


,shot_key,velocity_1,velocity_2,velocity_3,high_velocity,cylinder_pressure,rapid_rise_time,biscuit_thickness,clamping_force,cycle_time,pressure_rise_time,casting_pressure,spray_time,spray_1_time,spray_2_time,velocity_1_outlier_flag,velocity_2_outlier_flag,velocity_3_outlier_flag,high_velocity_outlier_flag,cylinder_pressure_outlier_flag,rapid_rise_time_outlier_flag,biscuit_thickness_outlier_flag,clamping_force_outlier_flag,cycle_time_outlier_flag,pressure_rise_time_outlier_flag,casting_pressure_outlier_flag,spray_time_outlier_flag,spray_1_time_outlier_flag,spray_2_time_outlier_flag
0,4207011_11,0.156,0.166,0.192,2.723,265,0.012,20,357,36.6,0.041,595,12.5,2.0,2.2,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,4208012_12,0.157,0.166,0.204,2.730,264,0.014,19,359,36.5,0.040,594,12.5,2.0,2.2,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,4209013_13,0.156,0.170,0.204,2.715,265,0.012,18,361,36.5,0.041,595,12.5,2.0,2.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4210014_14,0.154,0.170,0.202,2.717,264,0.011,20,364,36.5,0.042,595,12.5,2.0,2.2,0,0,0,0,1,0,0,0,0,0,0,0,0,0
4,4211015_15,0.146,0.160,0.198,2.684,264,0.012,20,357,36.5,0.042,595,12.5,2.0,2.2,0,0,0,0,1,0,0,0,0,0,0,0,0,0
5,4212016_16,0.154,0.174,0.204,2.699,264,0.012,20,359,36.4,0.042,594,12.5,2.0,2.2,0,0,0,0,1,0,0,0,0,0,0,0,0,0
6,4213017_17,0.158,0.166,0.206,2.678,265,0.011,18,363,36.5,0.042,596,12.5,2.0,2.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,4214018_18,0.160,0.162,0.204,2.703,264,0.011,18,363,36.4,0.042,594,12.5,2.0,2.2,0,0,0,0,1,0,0,0,0,0,0,0,0,0
8,4215019_19,0.154,0.170,0.205,2.669,265,0.011,18,354,36.4,0.042,595,12.5,2.0,2.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,4216020_20,0.158,0.164,0.205,2.700,264,0.014,18,355,36.3,0.041,594,12.5,2.0,2.2,0,0,0,0,1,0,0,0,0,0,0,0,0,0


<제품 2의 Sensor 관련 변수의 데이터프레임(컬럼추가)>


,melting_furnace_temp,air_pressure,coolant_temp,coolant_pressure,factory_temp,factory_humidity,melting_furnace_temp_outlier_flag,air_pressure_outlier_flag,coolant_temp_outlier_flag,coolant_pressure_outlier_flag,factory_temp_outlier_flag,factory_humidity_outlier_flag
0,671.6,6.5,26.1,2.70,33.2,57.4,0,0,0,0,0,0
1,672.1,6.4,26.2,2.71,33.3,57.0,0,0,0,0,0,0
2,672.4,6.4,26.2,2.70,33.5,56.7,0,0,0,0,0,0
3,672.4,6.4,26.2,2.70,33.5,56.7,0,0,0,0,0,0
4,672.4,6.3,26.2,2.71,33.6,56.4,0,0,0,0,0,0
5,672.5,6.3,26.2,2.69,33.7,56.2,0,0,0,0,0,0
6,672.5,6.3,26.2,2.69,33.7,56.2,0,0,0,0,0,0
7,672.4,6.2,26.2,2.71,33.8,56.1,0,0,0,0,0,0
8,672.4,6.2,26.2,2.71,33.8,56.1,0,0,0,0,0,0
9,671.2,6.1,26.2,2.71,33.9,56.0,0,0,0,0,0,0


#### 2.5.3 이상치가 존재하는 컬럼 확인
- 이상치가 1개 이상 존재하는 컬럼 외에는 플래깅 컬럼 삭제

In [22]:
# 1. _outlier_flag 관련 컬럼만 추출해서 이상치 개수 확인
def summarize_outliers(df_flagged, df_name):
    flag_cols = [c for c in df_flagged.columns if c.endswith("_outlier_flag")]
    
    total_rows = len(df_flagged)
    outlier_counts = df_flagged[flag_cols].sum().sort_values(ascending=False)
    outlier_ratio = (outlier_counts / total_rows * 100).round(2)
    
    summary = pd.DataFrame({
        '이상치 개수': outlier_counts,
        '비율(%)': outlier_ratio
    })
    summary.index = summary.index.str.replace("_outlier_flag", "", regex=False)
    # regex=False: _outlier_flag가 정규식이 아님을 명시하고, pandas의 경고메세지 예방용
    
    print(f"{'='*30}")
    print(f"<{df_name} 관련 데이터의 이상치 개수 및 비율 (전체 {total_rows}행 기준)>")
    print(f"{'='*30}")
    display(summary)

summarize_outliers(df_flagged_process_2, "제품 2의 Process")
summarize_outliers(df_flagged_sensor_2, "제품 2의 Sensor")

<제품 2의 Process 관련 데이터의 이상치 개수 및 비율 (전체 3328행 기준)>


,이상치 개수,비율(%)
cycle_time,763,22.93
cylinder_pressure,689,20.70
spray_1_time,136,4.09
rapid_rise_time,120,3.61
casting_pressure,49,1.47
biscuit_thickness,23,0.69
velocity_3,15,0.45
velocity_1,2,0.06
spray_time,1,0.03
velocity_2,0,0.00


<제품 2의 Sensor 관련 데이터의 이상치 개수 및 비율 (전체 3328행 기준)>


,이상치 개수,비율(%)
factory_temp,2,0.06
melting_furnace_temp,0,0.00
air_pressure,0,0.00
coolant_temp,0,0.00
coolant_pressure,0,0.00
factory_humidity,0,0.00


In [23]:
# 2. 이상치가 존재하지 않는 컬럼의 flagging 컬럼 삭제
# 2-1. 함수 정의
def drop_zero_outlier_flags(df):
    cols_to_drop = [
        col for col in df.columns 
        if col.endswith('_outlier_flag') and df[col].sum() == 0
    ]
    
    df_result = df.drop(columns=cols_to_drop)
    
    print(f"삭제된 플래그 컬럼 수: {len(cols_to_drop)}개")
    print(f"남은 플래그 컬럼(이상치가 존재하는 컬럼): {[c for c in df_result.columns if '_outlier_flag' in c]}")
    
    return df_result

In [24]:
# 2-2. Process 관련 데이터의 함수 실행 결과
df_fin_flag_process_2 = drop_zero_outlier_flags(df_flagged_process_2)

삭제된 플래그 컬럼 수: 5개
남은 플래그 컬럼(이상치가 존재하는 컬럼): ['velocity_1_outlier_flag', 'velocity_3_outlier_flag', 'cylinder_pressure_outlier_flag', 'rapid_rise_time_outlier_flag', 'biscuit_thickness_outlier_flag', 'cycle_time_outlier_flag', 'casting_pressure_outlier_flag', 'spray_time_outlier_flag', 'spray_1_time_outlier_flag']


In [25]:
# 2-3. Sensor 관련 데이터의 함수 실행 결과
df_fin_flag_sensor_2 = drop_zero_outlier_flags(df_flagged_sensor_2)

삭제된 플래그 컬럼 수: 5개
남은 플래그 컬럼(이상치가 존재하는 컬럼): ['factory_temp_outlier_flag']


In [26]:
# 2-4. 위 작업이 잘 진행되었는지 확인할 수 있는 코드 (주석 해지시 확인 가능)
#print(len(df_flagged_process_2.columns))
#print(len(df_flagged_sensor_2.columns))
#print('='*30)
#print(len(df_fin_flag_process_2.columns))
#print(len(df_fin_flag_sensor_2.columns))

#### 2.5.4 클래핑
- 이상치로 정의된 값은 이상치 정의 방법과 그에 따른 경계값에 따라 다음과 같이 이상치를 대체함. 
    - 하한 경계값보다 작은 값은 하한 경계값으로 대체하고 상한 경계값보다 큰 값은 상한 경계값으로 대체함

In [27]:
# 1. 함수 정의
def clamp_hybrid(df, skew_holder=2, kurt_holder=8):
    """
    abs(왜도) < 2 AND abs(첨도) < 8 → Z-score 3σ 방식으로 클램핑
    둘 중 하나라도 미충족 → IQR 방식으로 클램핑
    """

    df_clamped = df.copy()

    numeric_cols = df.select_dtypes(include=["number"]).columns # 숫자형 컬럼만 선택

    for column in numeric_cols:
        skew_value = df[column].skew()
        kurt_value = df[column].kurt()

        # Z-score; 3σ 방식
        if abs(skew_value) < skew_holder and abs(kurt_value) < kurt_holder:
            mean = df[column].mean()
            std = df[column].std(ddof=0)

            lower_bound = mean - 3 * std # 하한 경계값
            upper_bound = mean + 3 * std # 상한 경계값

        # IQR 방식
        else:
            Q1 = df[column].quantile(0.25)
            Q3 = df[column].quantile(0.75)
            IQR = Q3 - Q1

            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

        # 클램핑 적용
        df_clamped[column] = df[column].clip(lower=lower_bound, upper=upper_bound)

    return df_clamped

In [28]:
# 2. 원본에서 클램핑된 값 만들기 (Process 관련 데이터)
df_process_clamped = clamp_hybrid(df_cleaned_process_2)

# 2-1. 플래그 컬럼만 clamped에 붙이기
flag_cols = df_fin_flag_process_2.filter(like="_outlier_flag").columns
df_final_process_2 = pd.concat([df_process_clamped, df_fin_flag_process_2[flag_cols]], axis=1)

# 2-2. 결과 확인
df_final_process_2.head(5)

,shot_key,velocity_1,velocity_2,velocity_3,high_velocity,cylinder_pressure,rapid_rise_time,biscuit_thickness,clamping_force,cycle_time,pressure_rise_time,casting_pressure,spray_time,spray_1_time,spray_2_time,velocity_1_outlier_flag,velocity_3_outlier_flag,cylinder_pressure_outlier_flag,rapid_rise_time_outlier_flag,biscuit_thickness_outlier_flag,cycle_time_outlier_flag,casting_pressure_outlier_flag,spray_time_outlier_flag,spray_1_time_outlier_flag
0,4207011_11,0.156,0.166,0.192,2.723,265,0.012,20,357,36.55,0.041,595.0,12.5,2.0,2.2,0,0,0,0,0,1,0,0,0
1,4208012_12,0.157,0.166,0.204,2.730,265,0.014,19,359,36.50,0.040,594.0,12.5,2.0,2.2,0,0,1,0,0,0,0,0,0
2,4209013_13,0.156,0.170,0.204,2.715,265,0.012,18,361,36.50,0.041,595.0,12.5,2.0,2.2,0,0,0,0,0,0,0,0,0
3,4210014_14,0.154,0.170,0.202,2.717,265,0.011,20,364,36.50,0.042,595.0,12.5,2.0,2.2,0,0,1,0,0,0,0,0,0
4,4211015_15,0.146,0.160,0.198,2.684,265,0.012,20,357,36.50,0.042,595.0,12.5,2.0,2.2,0,0,1,0,0,0,0,0,0


In [29]:
# 3. 원본에서 클램핑된 값 만들기 (Sensor 관련 데이터)
df_sensor_clamped = clamp_hybrid(df_cleaned_sensor_2)

# 3-1. 플래그 컬럼만 clamped에 붙이기
flag_cols = df_fin_flag_sensor_2.filter(like="_outlier_flag").columns
df_final_sensor_2 = pd.concat([df_sensor_clamped, df_fin_flag_sensor_2[flag_cols]], axis=1)

# 3-2. 결과 확인
df_final_sensor_2.head(5)

,melting_furnace_temp,air_pressure,coolant_temp,coolant_pressure,factory_temp,factory_humidity,factory_temp_outlier_flag
0,671.6,6.5,26.1,2.70,33.2,57.4,0
1,672.1,6.4,26.2,2.71,33.3,57.0,0
2,672.4,6.4,26.2,2.70,33.5,56.7,0
3,672.4,6.4,26.2,2.70,33.5,56.7,0
4,672.4,6.3,26.2,2.71,33.6,56.4,0


---
### 2.6 전체 데이터에 대해 모든 전처리를 마친 데이터 파일로 저장

In [30]:
# 1. df_final_process_2, df_final_process_2, df_defects_groups_2가 통합된 데이터프레임 제작
df_final_overall_product_2 = pd.concat([df_final_process_2, df_final_sensor_2, df_defects_groups_2], axis=1)

print("shape:", df_final_overall_product_2.shape)
df_final_overall_product_2.head(5)

shape: (3328, 34)


,shot_key,velocity_1,velocity_2,velocity_3,high_velocity,cylinder_pressure,rapid_rise_time,biscuit_thickness,clamping_force,cycle_time,pressure_rise_time,casting_pressure,spray_time,spray_1_time,spray_2_time,velocity_1_outlier_flag,velocity_3_outlier_flag,cylinder_pressure_outlier_flag,rapid_rise_time_outlier_flag,biscuit_thickness_outlier_flag,cycle_time_outlier_flag,casting_pressure_outlier_flag,spray_time_outlier_flag,spray_1_time_outlier_flag,melting_furnace_temp,air_pressure,coolant_temp,coolant_pressure,factory_temp,factory_humidity,factory_temp_outlier_flag,surface_defect,structural_defect,contamination_defect
0,4207011_11,0.156,0.166,0.192,2.723,265,0.012,20,357,36.55,0.041,595.0,12.5,2.0,2.2,0,0,0,0,0,1,0,0,0,671.6,6.5,26.1,2.70,33.2,57.4,0,0,0,0
1,4208012_12,0.157,0.166,0.204,2.730,265,0.014,19,359,36.50,0.040,594.0,12.5,2.0,2.2,0,0,1,0,0,0,0,0,0,672.1,6.4,26.2,2.71,33.3,57.0,0,0,0,0
2,4209013_13,0.156,0.170,0.204,2.715,265,0.012,18,361,36.50,0.041,595.0,12.5,2.0,2.2,0,0,0,0,0,0,0,0,0,672.4,6.4,26.2,2.70,33.5,56.7,0,0,0,0
3,4210014_14,0.154,0.170,0.202,2.717,265,0.011,20,364,36.50,0.042,595.0,12.5,2.0,2.2,0,0,1,0,0,0,0,0,0,672.4,6.4,26.2,2.70,33.5,56.7,0,0,0,0
4,4211015_15,0.146,0.160,0.198,2.684,265,0.012,20,357,36.50,0.042,595.0,12.5,2.0,2.2,0,0,1,0,0,0,0,0,0,672.4,6.3,26.2,2.71,33.6,56.4,0,0,0,0


In [31]:
# 2. 통계를 위한 csv 파일로 저장 
df_final_overall_product_2.to_csv("../common-file/for_통계_overall_data_product-2.csv", index=False)
df_final_process_2.to_csv("../common-file/for_통계_process_data_product-2.csv", index=False)
df_final_sensor_2.to_csv("../common-file/for_통계_sensor_data_product-2.csv", index=False)
df_defects_groups_2.to_csv("../common-file/for_통계_defects_data_product-2.csv", index=False)
